# M22 Discovery Entrypoint — Telco Dataset Authoring Notebook

Telco Customer Churn authoring entrypoint for Atlas DataFlow (Project Spec S0011).

This notebook is the Telco-specific authoring surface for
`data/raw/telco-customer-churn.csv`. It loads the raw CSV, verifies known
structural facts, and records authoring observations that a later, separately
authorized modeling-intent contract spec can consume.

## Boundaries

- This notebook is an **authoring notebook only**.
- Notebook output is **not** the final operational source of truth.
- This notebook does not train models, select model families, create release
  candidates, validate publisher candidates, promote releases, mutate
  registry state, or change API/UI behavior.
- Any local output produced by this notebook is a **non-promoted authoring
  artifact**. It must not be treated as an official contract, release
  candidate, publisher run, registry file, model binary, or UI data fixture.
- Downstream contract derivation, model training, and publication are
  separate, later stages driven by separate, explicitly authorized specs.

## Usage

Run locally with the default repository-relative parameters, or override
`repo_root` explicitly (for example under papermill) if the notebook is
executed from outside the repository working directory:

```
papermill notebooks/m22_discovery_entrypoint.ipynb output.ipynb \
    -p repo_root /path/to/atlas-dataflow
```

Do not rely on implicit notebook state or hidden local paths as inputs to
later pipeline stages.

In [1]:
# Side-effect boundary — these operations are explicitly forbidden in this
# authoring entrypoint. This is checked before any dataset loading occurs.
FORBIDDEN_SIDE_EFFECTS = {
    "model_training": False,
    "model_family_selection": False,
    "release_candidate_creation": False,
    "publisher_run_creation": False,
    "release_promotion": False,
    "registry_state_mutation": False,
    "api_behavior_change": False,
    "ui_behavior_change": False,
    "reusable_helper_module_creation": False,
    "notebook_output_committed_as_official_artifact": False,
}
assert all(not v for v in FORBIDDEN_SIDE_EFFECTS.values()), (
    "Side-effect boundary violated; this entrypoint must not perform any "
    "forbidden operation."
)
print("Side-effect boundaries confirmed:", FORBIDDEN_SIDE_EFFECTS)

Side-effect boundaries confirmed: {'model_training': False, 'model_family_selection': False, 'release_candidate_creation': False, 'publisher_run_creation': False, 'release_promotion': False, 'registry_state_mutation': False, 'api_behavior_change': False, 'ui_behavior_change': False, 'reusable_helper_module_creation': False, 'notebook_output_committed_as_official_artifact': False}


## Parameters and repository-relative dataset path

`dataset_relative_path` is repository-relative and explicit — no implicit or
hidden absolute paths. `repo_root` defaults to the current working directory
(the expected convention when running this notebook from the repository
root) and may be overridden explicitly, for example by papermill, when the
notebook is executed from elsewhere.

In [3]:
# Parameters — supply explicit overrides here or via papermill; the defaults
# below are the Telco authoring defaults for this notebook.
dataset_slug = "telco-customer-churn"          # Fixed authoring identity for this notebook.
dataset_relative_path = "data/raw/telco-customer-churn.csv"  # Repository-relative, explicit.
repo_root = None                                # Optional override (str); None = current working directory.
target_column = "Churn"                         # Observed target column for this dataset.

## Raw CSV loading

Resolve the repository-relative dataset path against `repo_root` (or the
current working directory when `repo_root` is not supplied) and load the raw
CSV with the standard library `csv` module only. This notebook intentionally
avoids new repository dependencies (for example `pandas`) — none are
authorized by this spec.

In [4]:
import csv
import json
from collections import Counter
from pathlib import Path

_repo_root = Path(repo_root) if repo_root else Path.cwd()
dataset_path = (_repo_root / dataset_relative_path).resolve()

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {dataset_path}. "
        f"Expected the official raw CSV at repository-relative path "
        f"'{dataset_relative_path}' under repo_root '{_repo_root}'. "
        "Supply an explicit repo_root override if running outside the "
        "repository working directory."
    )

with dataset_path.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    raw_columns = list(reader.fieldnames or [])
    raw_rows = list(reader)

print(f"dataset_slug   : {dataset_slug}")
print(f"dataset_path   : {dataset_path}")
print(f"row_count      : {len(raw_rows)}")
print(f"column_count   : {len(raw_columns)}")

FileNotFoundError: Dataset not found at: /home/fabyuu/Projetos/N8N/atlas-dataflow/notebooks/data/raw/telco-customer-churn.csv. Expected the official raw CSV at repository-relative path 'data/raw/telco-customer-churn.csv' under repo_root '/home/fabyuu/Projetos/N8N/atlas-dataflow/notebooks'. Supply an explicit repo_root override if running outside the repository working directory.

## Structural verification

Verify the row count, column count, ordered column list, and raw dtypes
observed for the official Telco Customer Churn CSV. These are structural
facts recorded by this Project Spec (S0011) against the committed source
file. A mismatch means the source CSV changed since this notebook's
observations were authored and must be reviewed explicitly before trusting
the rest of this notebook's recorded observations.

In [ ]:
EXPECTED_ROW_COUNT = 7043
EXPECTED_COLUMN_COUNT = 21
EXPECTED_COLUMNS = [
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents",
    "tenure", "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling",
    "PaymentMethod", "MonthlyCharges", "TotalCharges", "Churn",
]

assert len(raw_rows) == EXPECTED_ROW_COUNT, (
    f"Row count {len(raw_rows)} does not match the recorded authoring "
    f"observation ({EXPECTED_ROW_COUNT}). The source CSV may have changed; "
    "revisit this notebook's recorded structural observations before "
    "trusting downstream sections."
)
assert len(raw_columns) == EXPECTED_COLUMN_COUNT, (
    f"Column count {len(raw_columns)} does not match the recorded "
    f"authoring observation ({EXPECTED_COLUMN_COUNT})."
)
assert raw_columns == EXPECTED_COLUMNS, (
    "Ordered column list does not match the recorded authoring observation. "
    f"Observed: {raw_columns}"
)


def classify_raw_dtype(column):
    """Classify a column as numeric_parseable or categorical from raw string values."""
    values = [row[column] for row in raw_rows]
    non_blank = [v for v in values if v.strip() != ""]
    numeric_parseable = 0
    for v in non_blank:
        try:
            float(v)
            numeric_parseable += 1
        except ValueError:
            pass
    all_numeric = non_blank and numeric_parseable == len(non_blank)
    return {
        "raw_dtype": "numeric_parseable" if all_numeric else "categorical",
        "blank_count": len(values) - len(non_blank),
    }


raw_dtypes = {column: classify_raw_dtype(column) for column in raw_columns}
print("Structural verification passed: row_count, column_count, ordered columns match.")
print("Raw dtypes (from CSV strings, before any coercion):")
print(json.dumps(raw_dtypes, indent=2))

## Target-column inspection — `Churn`

Inspect the observed target column labels and distribution. `Churn` is the
binary target column for this dataset. The likely positive-class candidate
for later modeling review is `Yes`, but the final positive-label decision
must be externalized by a later modeling-intent contract spec — this
notebook only records the observation, it does not decide it.

In [ ]:
EXPECTED_TARGET_LABELS = {"No", "Yes"}
EXPECTED_TARGET_DISTRIBUTION = {"No": 5174, "Yes": 1869}
LIKELY_POSITIVE_CLASS_CANDIDATE = "Yes"  # Observation only — not a final decision.

target_values = [row[target_column] for row in raw_rows]
target_label_set = set(target_values)
target_distribution = dict(Counter(target_values))

assert target_label_set == EXPECTED_TARGET_LABELS, (
    f"Observed target labels {sorted(target_label_set)} do not match the "
    f"recorded authoring observation {sorted(EXPECTED_TARGET_LABELS)}."
)
assert target_distribution == EXPECTED_TARGET_DISTRIBUTION, (
    f"Observed target distribution {target_distribution} does not match the "
    f"recorded authoring observation {EXPECTED_TARGET_DISTRIBUTION}. The "
    "source CSV may have changed; revisit this notebook's recorded "
    "observations before trusting downstream sections."
)

print(f"target_column                  : {target_column}")
print(f"observed_target_labels         : {sorted(target_label_set)}")
print(f"observed_target_distribution   : {target_distribution}")
print(f"likely_positive_class_candidate: {LIKELY_POSITIVE_CLASS_CANDIDATE} (observation only, not a final decision)")

## Identifier-column inspection — `customerID`

`customerID` is a per-row identifier candidate, not a modeling feature
candidate. It must be excluded from initial feature candidates without an
explicit override recorded elsewhere.

In [ ]:
IDENTIFIER_COLUMNS = ["customerID"]

customer_ids = [row["customerID"] for row in raw_rows]
customer_id_unique_count = len(set(customer_ids))

assert customer_id_unique_count == len(raw_rows), (
    f"Expected 'customerID' to be unique per row ({len(raw_rows)} rows), "
    f"observed {customer_id_unique_count} unique values. An identifier "
    "candidate that is not unique per row must be reviewed explicitly "
    "before being treated as an identifier."
)

print(f"identifier_columns       : {IDENTIFIER_COLUMNS}")
print(f"customerID_unique_count  : {customer_id_unique_count} (of {len(raw_rows)} rows)")
print("customerID is recorded as an identifier candidate and is excluded from "
      "initial feature candidates without explicit override.")

## Missing and blank-value inspection — `TotalCharges`

Surface the `TotalCharges` blank-string condition explicitly rather than
silently coercing it to numeric. Blank-value handling for `TotalCharges`
must be decided explicitly by a later modeling-intent contract spec before
final modeling intent — this notebook only records the observation.

In [ ]:
total_charges_blank_rows = [
    row for row in raw_rows if row["TotalCharges"].strip() == ""
]
total_charges_blank_count = len(total_charges_blank_rows)
total_charges_blank_tenure_values = sorted(
    {row["tenure"] for row in total_charges_blank_rows}
)

if total_charges_blank_count == 0:
    print(
        "No blank 'TotalCharges' values observed in this run of the CSV. "
        "This differs from this notebook's recorded authoring observation "
        "(11 blank values, all at tenure == '0') — revisit before trusting "
        "downstream authoring notes."
    )
else:
    print(f"total_charges_blank_count           : {total_charges_blank_count}")
    print(f"total_charges_blank_tenure_values   : {total_charges_blank_tenure_values}")

print(
    "'TotalCharges' must not be silently treated as fully numeric. Blank-value "
    "handling (for example impute-as-zero, drop, or a distinct missing-value "
    "indicator) must be decided explicitly by a later modeling-intent "
    "contract spec, not implicitly by this authoring notebook or by "
    "downstream training code."
)

## Feature-candidate overview

List non-target, non-identifier columns as initial feature candidates only.
`SeniorCitizen` is recorded separately as a binary numeric indicator, since
it is already represented as `0`/`1` in the raw CSV rather than as a raw
categorical label.

In [ ]:
EXCLUDED_FROM_FEATURE_CANDIDATES = set(IDENTIFIER_COLUMNS) | {target_column}

feature_candidate_columns = [
    column for column in raw_columns if column not in EXCLUDED_FROM_FEATURE_CANDIDATES
]

senior_citizen_values = sorted({row["SeniorCitizen"] for row in raw_rows})
assert set(senior_citizen_values) == {"0", "1"}, (
    f"Expected 'SeniorCitizen' to be a binary numeric indicator with "
    f"observed values {{'0', '1'}}, observed {senior_citizen_values}."
)

print(f"excluded_from_feature_candidates : {sorted(EXCLUDED_FROM_FEATURE_CANDIDATES)}")
print(f"feature_candidate_columns        : {feature_candidate_columns}")
print(f"SeniorCitizen_observed_values    : {senior_citizen_values} (binary numeric indicator)")
print(
    "These are initial feature candidates only — final feature selection, "
    "encoding, and missing-value policy are decided by a later "
    "modeling-intent contract spec, not by this authoring notebook."
)

## Preliminary authoring observations

Assemble the authoring observations recorded by this notebook into a single
structure. This is an authoring-time observation record intended to feed a
later, separately authorized modeling-intent contract spec — it is not
itself an execution contract, release candidate, or any other governed
pipeline artifact.

In [ ]:
authoring_observations = {
    "dataset_slug": dataset_slug,
    "dataset_relative_path": dataset_relative_path,
    "row_count": len(raw_rows),
    "column_count": len(raw_columns),
    "ordered_columns": raw_columns,
    "raw_dtypes": raw_dtypes,
    "target_column": target_column,
    "observed_target_labels": sorted(target_label_set),
    "observed_target_distribution": target_distribution,
    "likely_positive_class_candidate": LIKELY_POSITIVE_CLASS_CANDIDATE,
    "positive_class_decision_finalized": False,
    "identifier_columns": IDENTIFIER_COLUMNS,
    "total_charges_blank_count": total_charges_blank_count,
    "total_charges_blank_value_policy_decided": False,
    "feature_candidate_columns": feature_candidate_columns,
    "senior_citizen_observed_values": senior_citizen_values,
    "notebook_boundary": "authoring_notebook_only",
    "notebook_output_is_not_final_operational_truth": True,
    "requires_later_modeling_intent_contract_spec": True,
}

print("Authoring observations:")
print(json.dumps(authoring_observations, indent=2))

## Non-promoted local output summary

This notebook may optionally write `authoring_observations` to a local file
for the operator's own convenience during authoring. Any such file is a
**non-promoted authoring artifact** — it lives under `data/`, which is
excluded from version control by this repository's `.gitignore`, and it must
not be treated as an official contract, release candidate, publisher run,
registry file, or UI data fixture. Promoting any of these observations into
an official artifact requires a separate, explicitly authorized
implementation request.

In [ ]:
write_local_output = False  # Set True locally to write the non-promoted summary file below.

local_output_dir = _repo_root / "data" / "raw" / "telco-customer-churn-authoring-output"
local_output_path = local_output_dir / f"{dataset_slug}-authoring-observations.json"

if write_local_output:
    local_output_dir.mkdir(parents=True, exist_ok=True)
    local_output_path.write_text(
        json.dumps(authoring_observations, indent=2), encoding="utf-8"
    )
    print(f"Non-promoted authoring output written to: {local_output_path}")
else:
    print(
        "write_local_output is False; no local output file was written. "
        f"If enabled, the non-promoted authoring output would be written to: "
        f"{local_output_path}"
    )

print(
    "This output path is local and non-promoted. It is not an official "
    "contract, release candidate, publisher run, registry file, or UI data "
    "fixture. Promotion requires a separate, explicitly authorized "
    "implementation request."
)